In [ ]:
import requests
import json
import re
from pydantic import BaseModel

In [ ]:
BASE_URL = "http://localhost:8000"

In [ ]:
class Usage(BaseModel):
    """Standardized Pydantic model for token usage and cost."""
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    cost_usd: float


def print_usage(usage: Usage):
    """Standard function to print formatted token metrics and cost."""
    print("\n" + "-" * 42)
    print("📊 TOKEN USAGE & COST SUMMARY")
    print(f"  • Prompt Tokens:     {usage.prompt_tokens}")
    print(f"  • Completion Tokens: {usage.completion_tokens}")
    print(f"  • Total Tokens:      {usage.total_tokens}")
    print(f"  • Estimated Cost:    ${usage.cost_usd:.6f}")
    print("-" * 42)

In [ ]:
# ==========================================
# Standard GET Request Test Function
# ==========================================
def test_get_standard(query):
    """Test standard GET request."""  # Function docstring summarizing the test's purpose
    
    # Print a clear section header to the terminal output for readability
    print("=== 1. Testing GET (Standard) ===")
    
    # Construct the full target URL for the GET generation endpoint
    url = f"{BASE_URL}/generate/get"
    # Define query parameters dictionary containing the user's prompt query
    params = {"query": query}
    
    # Send the HTTP GET request to the FastAPI server with query parameters
    response = requests.get(url, params=params)
    
    # Check if the request was successful (HTTP status code 200 OK)
    if response.status_code == 200:
        # Parse the JSON response body into a Python dictionary
        data = response.json()
        
        # Print the core AI-generated text response returned by the server
        print(f"Response:\n{data.get('response')}")
        
        # Check if token usage telemetry was included in the server response payload
        if data.get("usage"):
            # Unpack the usage dictionary into a Pydantic 'Usage' model for validation & formatting
            usage_obj = Usage(**data["usage"])
            # Call a helper print function to nicely display token counts and cost USD
            print_usage(usage_obj)
            
    else:
        # If the request failed, print the error status code and response detail message
        print(f"Error {response.status_code}: {response.text}")
        
    # Print a trailing divider line to separate test logs in the terminal
    print("\n" + "="*45 + "\n")

In [ ]:
# ==========================================
# Streaming GET Request Test Function
# ==========================================
def test_get_streaming(query):
    """Test streaming GET request with clean output filtering."""
    print("=== 2. Testing GET (Streaming) ===")
    url = f"{BASE_URL}/generate/get"
    params = {"query": query, "stream": "true"}
    
    full_output = ""
    with requests.get(url, params=params, stream=True) as response:
        if response.status_code == 200:
            print("Response: ", end="")
            for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
                if chunk:
                    full_output += chunk
                    
                    # Check if the chunk contains the usage stats string
                    if "[Usage Stats]" in chunk:
                        # Split out the usage part so it doesn't print raw to the screen
                        clean_text = chunk.split("[Usage Stats]")[0]
                        print(clean_text, end="", flush=True)
                    else:
                        print(chunk, end="", flush=True)
            print("\n")
            
            # Extract usage stats from the full buffer and display via print_usage
            match = re.search(
                r"\[Usage Stats\] Prompt Tokens: (\d+) \| Completion Tokens: (\d+) \| Total Tokens: (\d+) \| Estimated Cost: \$([\d\.]+)",
                full_output
            )
            if match:
                usage_obj = Usage(
                    prompt_tokens=int(match.group(1)),
                    completion_tokens=int(match.group(2)),
                    total_tokens=int(match.group(3)),
                    cost_usd=float(match.group(4))
                )
                print_usage(usage_obj)
        else:
            print(f"Error {response.status_code}: {response.text}")
    print("="*45 + "\n")

In [ ]:
# ==========================================
# Standard POST Request Test Function
# ==========================================
def test_post_standard(query):
    """Test standard POST request."""  # Function docstring summarizing the test's objective
    
    # Print a clear section header to the terminal for the standard POST test run
    print("=== 3. Testing POST (Standard) ===")
    
    # Construct the target URL for the POST generation endpoint
    url = f"{BASE_URL}/generate/post"
    
    # Define the request body dictionary mapping to the FastAPI 'Question' Pydantic model
    payload = {"query": query}
    
    # Send an HTTP POST request to the server, automatically serializing the payload dictionary to JSON
    response = requests.post(url, json=payload)
    
    # Verify that the request completed successfully with an HTTP 200 OK status code
    if response.status_code == 200:
        # Parse the JSON response body into a Python dictionary
        data = response.json()
        
        # Print the AI model's generated response text
        print(f"Response:\n{data.get('response')}")
        
        # Check if token telemetry usage metrics were returned in the response payload
        if data.get("usage"):
            # Unpack the usage dictionary into the Pydantic 'Usage' model
            usage_obj = Usage(**data["usage"])
            # Display formatted token counts and financial cost
            print_usage(usage_obj)
            
    else:
        # Handle non-successful status codes by printing the error details
        print(f"Error {response.status_code}: {response.text}")
        
    # Print a trailing divider line to separate console output blocks
    print("\n" + "="*45 + "\n")

In [ ]:
# ==========================================
# Streaming POST Request Test Function
# ==========================================
def test_post_streaming(query):
    """Test streaming POST request with clean output filtering."""
    print("=== 4. Testing POST (Streaming) ===")
    url = f"{BASE_URL}/generate/post"
    params = {"stream": "true"}
    payload = {"query": query}
    
    full_output = ""
    # Open an HTTP POST request with streaming enabled to handle incremental data delivery
    with requests.post(url, params=params, json=payload, stream=True) as response:
        # Verify that the server successfully processed the request with an HTTP 200 OK status
        if response.status_code == 200:
            print("Response: ", end="")

            # Iterate through incoming data chunks dynamically as they arrive from the server
            for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
                if chunk:
                    full_output += chunk
                    
                    # Filter out the raw usage string from printing directly to the terminal display
                    if "[Usage Stats]" in chunk:
                        # Split the chunk to isolate and hide the telemetry footer from live text output
                        clean_text = chunk.split("[Usage Stats]")[0]
                        print(clean_text, end="", flush=True)
                    else:
                        # Stream standard text content live to the terminal immediately with flush enabled
                        print(chunk, end="", flush=True)
            print("\n")
            
            # Search the accumulated output buffer using regex to extract token counts and financial cost
            match = re.search(
                r"\[Usage Stats\] Prompt Tokens: (\d+) \| Completion Tokens: (\d+) \| Total Tokens: (\d+) \| Estimated Cost: \$([\d\.]+)",
                full_output
            )
            
            # If telemetry match is found, parse the captured groups into the Pydantic Usage model
            if match:
                usage_obj = Usage(
                    prompt_tokens=int(match.group(1)),
                    completion_tokens=int(match.group(2)),
                    total_tokens=int(match.group(3)),
                    cost_usd=float(match.group(4))
                )
                # Display the structured token usage metrics and cost report in the terminal
                print_usage(usage_obj)
        else:
            print(f"Error {response.status_code}: {response.text}")
    print("="*45 + "\n")

In [ ]:
test_get_standard("Give me a fun fact about black holes")

In [ ]:
test_get_streaming("Write a limerick for me")

In [ ]:
test_post_standard("What is AI")

In [ ]:
test_post_streaming("What is RAG?")